# 02 — Model Training & Evaluation
Full two-stage pipeline: CV, tuning, stacking, SHAP, uncertainty, PSI.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, time, json, datetime
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from imblearn.combine import SMOTETomek

from src.utils.config import (DATA_DIR, SAMPLE_DIR, MODELS_DIR,
                               TEST_SIZE, RANDOM_STATE, CV_FOLDS)
from src.features.pipeline import (engineer_features, build_feature_matrix,
                                    get_preprocessor, DEMO_FEATURES, FULL_FEATURES)
from src.models.train import (get_models, train_with_cv, tune_random_forest,
                               build_stacking_ensemble, optimize_threshold)
from src.evaluation.metrics import (evaluate_model, monte_carlo_uncertainty,
                                     monitor_drift, run_shap_analysis)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi':110, 'font.size':11})
os.makedirs(MODELS_DIR, exist_ok=True)
print("✅ Imports OK")


In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
try:
    df = pd.read_csv(DATA_DIR / 'processed' / 'ASD_Combined_Enhanced_Final.csv')
    print(f"Full dataset: {df.shape}")
except FileNotFoundError:
    df = pd.read_csv(SAMPLE_DIR / 'ASD_sample_public.csv')
    print(f"⚠️  Using public sample: {df.shape}")

df = engineer_features(df)
print(f"After engineering: {df.shape}")
df.head(3)


In [ ]:
# ── Prepare feature matrices ─────────────────────────────────────────────────
def prepare_split(df, feature_cols, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    from sklearn.preprocessing import StandardScaler
    X = df[feature_cols].values.astype(float)
    y = df['Class'].values.astype(int)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size,
                                                random_state=random_state, stratify=y)
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)
    X_bal, y_bal = SMOTETomek(random_state=42).fit_resample(X_tr_sc, y_tr)
    print(f"  Train: {X_tr_sc.shape} | Test: {X_te_sc.shape} | Balanced: {X_bal.shape}")
    return X_tr_sc, X_te_sc, X_bal, y_bal, y_tr, y_te, scaler

print("=== DEMO split ===")
X_tr_d, X_te_d, X_bal_d, y_bal_d, y_tr_d, y_te_d, sc_d = prepare_split(df, DEMO_FEATURES)
print("\n=== FULL split ===")
X_tr_f, X_te_f, X_bal_f, y_bal_f, y_tr_f, y_te_f, sc_f = prepare_split(df, FULL_FEATURES)


In [ ]:
# ── Cross-validated training ───────────────────────────────────────────────
demo_models = get_models()
full_models = get_models()

_, _, cv_demo = train_with_cv(demo_models, X_bal_d, y_bal_d, label='DEMO')
_, _, cv_full = train_with_cv(full_models, X_bal_f, y_bal_f, label='FULL')


In [ ]:
# ── Hyperparameter tuning ──────────────────────────────────────────────────
print("Tuning RF for DEMO...")
tuned_rf_demo = tune_random_forest(X_bal_d, y_bal_d, label='DEMO')
demo_models['Tuned RF'] = tuned_rf_demo

print("\nTuning RF for FULL...")
tuned_rf_full = tune_random_forest(X_bal_f, y_bal_f, label='FULL')
full_models['Tuned RF'] = tuned_rf_full


In [ ]:
# ── Stacking ensembles ─────────────────────────────────────────────────────
print("Building stacking ensembles...")
stack_demo = build_stacking_ensemble(X_bal_d, y_bal_d, label='DEMO')
stack_full = build_stacking_ensemble(X_bal_f, y_bal_f, label='FULL')


In [ ]:
# ── Evaluate all models ────────────────────────────────────────────────────
print("\n=== DEMOGRAPHIC MODEL RESULTS ===")
eval_demo = {}
for name, model in {**demo_models, 'Stacking':stack_demo}.items():
    if not hasattr(model,'predict_proba'): continue
    model.fit(X_bal_d, y_bal_d)
    y_pred_opt, thresh, _ = optimize_threshold(model, X_te_d, y_te_d)
    eval_demo[name] = evaluate_model(model, X_te_d, y_te_d, name, y_pred_opt, thresh)

print("\n=== FULL AQ-10 MODEL RESULTS ===")
eval_full = {}
for name, model in {**full_models, 'Stacking':stack_full}.items():
    if not hasattr(model,'predict_proba'): continue
    model.fit(X_bal_f, y_bal_f)
    y_pred_opt, thresh, _ = optimize_threshold(model, X_te_f, y_te_f)
    eval_full[name] = evaluate_model(model, X_te_f, y_te_f, name, y_pred_opt, thresh)


## 📈 Visualizations

In [ ]:
# ── ROC Curves ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16,6))
colors = plt.cm.tab10(np.linspace(0, 0.8, max(len(eval_demo), len(eval_full))))

for (name, res), c in zip(eval_demo.items(), colors):
    fpr, tpr, _ = roc_curve(y_te_d, res['y_prob'])
    axes[0].plot(fpr, tpr, lw=2, color=c, label=f"{name} (AUC={res['roc_auc']:.3f})")
axes[0].plot([0,1],[0,1],'k--', lw=1); axes[0].set_title('Demographic Baseline', fontweight='bold')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

for (name, res), c in zip(eval_full.items(), colors):
    fpr, tpr, _ = roc_curve(y_te_f, res['y_prob'])
    axes[1].plot(fpr, tpr, lw=2, color=c, label=f"{name} (AUC={res['roc_auc']:.3f})")
axes[1].plot([0,1],[0,1],'k--', lw=1); axes[1].set_title('Full AQ-10 Model', fontweight='bold')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.suptitle('ROC Curves: Demographic Baseline vs Full AQ-10', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── Feature Importance ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16,6))
tuned_rf_demo.fit(X_bal_d, y_bal_d)
tuned_rf_full.fit(X_bal_f, y_bal_f)
fi_d = pd.Series(tuned_rf_demo.feature_importances_, index=DEMO_FEATURES).sort_values()
fi_f = pd.Series(tuned_rf_full.feature_importances_, index=FULL_FEATURES).sort_values()
axes[0].barh(fi_d.index, fi_d.values, color='#4C9BE8', edgecolor='white')
axes[0].set_title('Feature Importance — Demographic Model', fontweight='bold')
colors_full = ['#E8704C' if f.startswith('A') else '#4C9BE8' for f in fi_f.index]
axes[1].barh(fi_f.index, fi_f.values, color=colors_full, edgecolor='white')
axes[1].set_title('Feature Importance — Full AQ-10 (red=AQ, blue=demo)', fontweight='bold')
plt.suptitle('Feature Importance Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── SHAP Explainability ─────────────────────────────────────────────────────
import shap
result = run_shap_analysis(tuned_rf_full, X_bal_f, X_te_f, FULL_FEATURES, 'Full AQ-10 Model')
if result:
    sv, ev, clean_names = result
    fig, axes = plt.subplots(1, 2, figsize=(18,7))
    plt.sca(axes[0]); shap.summary_plot(sv, X_te_f, feature_names=clean_names, show=False, plot_type='bar')
    axes[0].set_title('SHAP Feature Importance', fontweight='bold')
    plt.sca(axes[1]); shap.summary_plot(sv, X_te_f, feature_names=clean_names, show=False)
    axes[1].set_title('SHAP Beeswarm', fontweight='bold')
    plt.suptitle('Global Explainability (SHAP)', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()


In [ ]:
# ── Uncertainty Quantification ──────────────────────────────────────────────
udf = monte_carlo_uncertainty(tuned_rf_full, X_te_f)
fig, axes = plt.subplots(1,2, figsize=(14,5))
sc = axes[0].scatter(udf['mean_probability'], udf['uncertainty_std'],
                      c=y_te_f, cmap='RdYlGn_r', alpha=0.5, s=20)
axes[0].axhline(0.15, color='red', linestyle='--', lw=1.5, label='Threshold')
axes[0].set_xlabel('Mean Probability'); axes[0].set_ylabel('Std (uncertainty)')
axes[0].set_title('Uncertainty Map', fontweight='bold'); axes[0].legend()
plt.colorbar(sc, ax=axes[0], label='True Class')
axes[1].hist(udf[udf['uncertain_flag']==False]['mean_probability'], bins=25, alpha=0.7, color='#4C9BE8', label='Certain', edgecolor='white')
axes[1].hist(udf[udf['uncertain_flag']==True]['mean_probability'],  bins=15, alpha=0.7, color='#E8704C', label='Uncertain', edgecolor='white')
axes[1].set_title('Prediction Distribution', fontweight='bold'); axes[1].legend()
plt.suptitle('Uncertainty Quantification', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── PSI Drift Detection ─────────────────────────────────────────────────────
np.random.seed(99)
n = len(X_te_f)
X_drift = X_te_f.copy()
sex_i = FULL_FEATURES.index('Sex') if 'Sex' in FULL_FEATURES else None
age_i = FULL_FEATURES.index('Age') if 'Age' in FULL_FEATURES else None
if sex_i: X_drift[:, sex_i] = np.random.choice([0,1], n, p=[0.65,0.35])
if age_i: X_drift[:, age_i] = (np.random.normal(35,15,n) - X_te_f[:,age_i if age_i else 0].mean()) / (X_te_f[:,age_i if age_i else 0].std()+1e-8)

drift_df = monitor_drift(X_tr_f, X_drift, FULL_FEATURES)
print("\n── Drift Report ──────────────────────────────────────────")
print(drift_df.to_string())

fig, ax = plt.subplots(figsize=(10, max(5, len(drift_df)*0.4+1)))
colors_ = ['#E8704C' if s=='🔴 RETRAIN' else ('#E8A74C' if s=='⚠️ MONITOR' else '#6ABF69') for s in drift_df['Status']]
ax.barh(drift_df.index, drift_df['PSI'], color=colors_, edgecolor='white')
ax.axvline(0.1, color='orange', linestyle='--', lw=1.5, label='Monitor')
ax.axvline(0.2, color='red',    linestyle='--', lw=1.5, label='Retrain')
ax.set_title('PSI Drift Detection (Demographic Shift Simulation)', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Save models ─────────────────────────────────────────────────────────────
import joblib
joblib.dump(tuned_rf_full, MODELS_DIR / 'asd_full_aq10_model.pkl')
joblib.dump(sc_f,          MODELS_DIR / 'asd_full_aq10_scaler.pkl')
joblib.dump(tuned_rf_demo, MODELS_DIR / 'asd_demographic_model.pkl')
joblib.dump(sc_d,          MODELS_DIR / 'asd_demographic_scaler.pkl')
print("✅ Models saved to models/ (excluded from git)")

best_demo_n = max(eval_demo, key=lambda k: eval_demo[k]['roc_auc'])
best_full_n = max(eval_full, key=lambda k: eval_full[k]['roc_auc'])
card = {
    'created': str(datetime.date.today()),
    'instrument': 'AQ-10 (Allison et al. 2012)',
    'note': 'Class label assigned by AQ-10 cutoff. Near-perfect AUC is expected.',
    'demographic_model': {'algorithm': best_demo_n, 'test_auc': round(eval_demo[best_demo_n]['roc_auc'],4)},
    'full_model':        {'algorithm': best_full_n, 'test_auc': round(eval_full[best_full_n]['roc_auc'],4)},
}
with open(MODELS_DIR / 'model_card.json', 'w') as f: json.dump(card, f, indent=2)
print("✅ Model card saved")


## Summary

| Stage | Best Model | Test AUC | Note |
|-------|-----------|----------|------|
| Demographic baseline | See output above | ≈ 0.65 | Pre-screening without questionnaire |
| Full AQ-10 model | See output above | ≈ 0.99 | AQ-10 validated on this dataset construction |

ΔAUC ≈ +0.34 — quantifies clinical value of AQ-10 instrument.
